# Day 4

## Tokenizing with code

In [27]:
from transformers import AutoTokenizer

# DeepSeek or Mistral models provide public tokenizers without gated access
tokenizer = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-v0.1")

text = "Hola, Soy Federico de la Vega y en noviembre voy a empezar a trabajar, mi dni es 45469668."

# Get integer token IDs (Equivalent to tiktoken encode)
token_ids = tokenizer.encode(text)


# Inspecting individual tokens/pieces
tokens = tokenizer.convert_ids_to_tokens(token_ids)

In [28]:
#Token IDs
print("Token IDs:", token_ids)

#Token ids to token
print("Token pieces:", tokens)

Token IDs: [1, 4170, 28708, 28725, 318, 904, 12969, 2224, 340, 543, 550, 4770, 337, 481, 26634, 15545, 264, 877, 19033, 283, 264, 23017, 283, 28725, 5192, 281, 3023, 1037, 28705, 28781, 28782, 28781, 28784, 28774, 28784, 28784, 28783, 28723]
Token pieces: ['<s>', '▁Hol', 'a', ',', '▁S', 'oy', '▁Feder', 'ico', '▁de', '▁la', '▁V', 'ega', '▁y', '▁en', '▁noviembre', '▁voy', '▁a', '▁em', 'pez', 'ar', '▁a', '▁trabaj', 'ar', ',', '▁mi', '▁d', 'ni', '▁es', '▁', '4', '5', '4', '6', '9', '6', '6', '8', '.']


In [29]:
for token_id in token_ids:
    token_text = tokenizer.convert_ids_to_tokens([token_id])
    print(f"{token_id} = {token_text}")

1 = ['<s>']
4170 = ['▁Hol']
28708 = ['a']
28725 = [',']
318 = ['▁S']
904 = ['oy']
12969 = ['▁Feder']
2224 = ['ico']
340 = ['▁de']
543 = ['▁la']
550 = ['▁V']
4770 = ['ega']
337 = ['▁y']
481 = ['▁en']
26634 = ['▁noviembre']
15545 = ['▁voy']
264 = ['▁a']
877 = ['▁em']
19033 = ['pez']
283 = ['ar']
264 = ['▁a']
23017 = ['▁trabaj']
283 = ['ar']
28725 = [',']
5192 = ['▁mi']
281 = ['▁d']
3023 = ['ni']
1037 = ['▁es']
28705 = ['▁']
28781 = ['4']
28782 = ['5']
28781 = ['4']
28784 = ['6']
28774 = ['9']
28784 = ['6']
28784 = ['6']
28783 = ['8']
28723 = ['.']


In [10]:
tokenizer.convert_ids_to_tokens([15359])

['▁Hi']

# And another topic!

### The Illusion of "memory"

Many of you will know this already. But for those that don't -- this might be an "AHA" moment!

In [15]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)
google_api_key = os.getenv('GOOGLE_API_KEY')

if not google_api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not google_api_key.startswith(("AIz", "AQ.")):
    print("An API key was found, but it doesn't start with AIz or AQ.")
else:
    print("API key found and looks good so far!")

API key found and looks good so far!


### You should be very comfortable with what the next cell is doing!

_I'm creating a new instance of the OpenAI Python Client library, a lightweight wrapper around making HTTP calls to an endpoint for calling the GPT LLM, or other LLM providers_

In [16]:
from openai import OpenAI

GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"

openai = OpenAI(
    base_url=GEMINI_BASE_URL, 
    api_key=google_api_key
)

### A message to OpenAI is a list of dicts

In [20]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "Hi! I'm Fede!"}
    ]

In [21]:
response = openai.chat.completions.create(model="gemini-3.5-flash-lite", messages=messages)
response.choices[0].message.content

"Hi Fede! It's great to meet you. How can I help you today?"

### OK let's now ask a follow-up question

In [22]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "What's my name?"}
    ]

In [23]:
response = openai.chat.completions.create(model="gemini-3.5-flash-lite", messages=messages)
response.choices[0].message.content

"I don't know your name yet! What should I call you?"

### Wait, wha??

We just told you!

What's going on??

Here's the thing: every call to an LLM is completely STATELESS. It's a totally new call, every single time. As AI engineers, it's OUR JOB to devise techniques to give the impression that the LLM has a "memory".

In [24]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "Hi! I'm Fede!"},
    {"role": "assistant", "content": "Hi Fede! How can I assist you today?"},
    {"role": "user", "content": "What's my name?"}
    ]

In [26]:
response = openai.chat.completions.create(model="gemini-3.5-flash-lite", messages=messages)
response.choices[0].message.content

'Your name is Fede!'

## To recap

With apologies if this is obvious to you - but it's still good to reinforce:

1. Every call to an LLM is stateless
2. We pass in the entire conversation so far in the input prompt, every time
3. This gives the illusion that the LLM has memory - it apparently keeps the context of the conversation
4. But this is a trick; it's a by-product of providing the entire conversation, every time
5. An LLM just predicts the most likely next tokens in the sequence; if that sequence contains "My name is Ed" and later "What's my name?" then it will predict.. Ed!

The ChatGPT product uses exactly this trick - every time you send a message, it's the entire conversation that gets passed in.

"Does that mean we have to pay extra each time for all the conversation so far"

For sure it does. And that's what we WANT. We want the LLM to predict the next tokens in the sequence, looking back on the entire conversation. We want that compute to happen, so we need to pay the electricity bill for it!

